# LA Crime / Big Data - EDA

Goal: understand each dataset's **structure, types, delimiters, quirks** so we read them correctly in Spark (`common/Datasets.java`).

Datasets (local copies pulled from HDFS into `../data/`, gitignored):

| file | size | format | key fields | used by |
|---|---|---|---|---|
| `LA_Crime_Data/*.csv` | ~900MB | quoted CSV `,` | `DATE OCC`,`TIME OCC`,`Premis Desc`,`AREA NAME`,`LAT`,`LON` | Q1,Q2,Q4 |
| `LA_income_2021.csv` | 11KB | CSV `;` | `Zip Code`,`Estimated Median Income` (`$` fmt) | Q3 |
| `LA_Census_Blocks_2020.geojson` | 171MB | GeoJSON (1 Feature/line) | `ZCTA20`,`POP20` | Q3 |
| `LA_Census_Blocks_2020_fields.csv` | 1.6KB | CSV `,` | data dictionary | ref |
| `LA_Police_Stations.csv` | 1.4KB | CSV `,` (BOM!) | `X`(lon),`Y`(lat),`DIVISION`,`PREC` | Q4 |

In [1]:
import json, os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

DATA = os.path.join('..', 'data')
CRIME_A = os.path.join(DATA, 'LA_Crime_Data', 'LA_Crime_Data_2010_2019.csv')
CRIME_B = os.path.join(DATA, 'LA_Crime_Data', 'LA_Crime_Data_2020_2025.csv')
INCOME  = os.path.join(DATA, 'LA_income_2021.csv')
GEOJSON = os.path.join(DATA, 'LA_Census_Blocks_2020.geojson')
FIELDS  = os.path.join(DATA, 'LA_Census_Blocks_2020_fields.csv')
STATIONS= os.path.join(DATA, 'LA_Police_Stations.csv')

for p in [CRIME_A, CRIME_B, INCOME, GEOJSON, FIELDS, STATIONS]:
    print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p}" if os.path.exists(p) else f"MISSING   {p}")

  638.15 MB  ../data/LA_Crime_Data/LA_Crime_Data_2010_2019.csv
  301.54 MB  ../data/LA_Crime_Data/LA_Crime_Data_2020_2025.csv
    0.01 MB  ../data/LA_income_2021.csv
  179.73 MB  ../data/LA_Census_Blocks_2020.geojson
    0.00 MB  ../data/LA_Census_Blocks_2020_fields.csv
    0.00 MB  ../data/LA_Police_Stations.csv


## 1. Crime data (Q1, Q2, Q4)

Two CSVs split by date range, same schema, comma-delimited, **all fields quoted**. We union them in Spark. Inspect a sample for dtypes; the columns we care about are text by default - we cast in the loader.

In [2]:
# Read a sample for structure (full file is ~600MB). dtype=str => see raw values, no silent coercion.
crime = pd.read_csv(CRIME_A, nrows=200_000, dtype=str)
print('shape (sample):', crime.shape)
print('\ncolumns:')
for c in crime.columns:
    print(' ', repr(c))

shape (sample): (200000, 28)

columns:
  'DR_NO'
  'Date Rptd'
  'DATE OCC'
  'TIME OCC'
  'AREA'
  'AREA NAME'
  'Rpt Dist No'
  'Part 1-2'
  'Crm Cd'
  'Crm Cd Desc'
  'Mocodes'
  'Vict Age'
  'Vict Sex'
  'Vict Descent'
  'Premis Cd'
  'Premis Desc'
  'Weapon Used Cd'
  'Weapon Desc'
  'Status'
  'Status Desc'
  'Crm Cd 1'
  'Crm Cd 2'
  'Crm Cd 3'
  'Crm Cd 4'
  'LOCATION'
  'Cross Street'
  'LAT'
  'LON'


In [3]:
crime.head(5)

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,001307355,2010 Feb 20 12:00:00 AM,2010 Feb 20 12:00:00 AM,1350,13,Newton,1385,2,900,VIOLATION OF COURT ORDER,0913 1814 2000,48,M,H,501,SINGLE FAMILY DWELLING,NaN,NaN,AA,Adult Arrest,900,NaN,NaN,NaN,300 E GAGE AV,NaN,33.9825,-118.2695
1,011401303,2010 Sep 13 12:00:00 AM,2010 Sep 12 12:00:00 AM,0045,14,Pacific,1485,2,740,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VA...",0329,0,M,W,101,STREET,NaN,NaN,IC,Invest Cont,740,NaN,NaN,NaN,SEPULVEDA BL,MANCHESTER AV,33.9599,-118.3962
2,070309629,2010 Aug 09 12:00:00 AM,2010 Aug 09 12:00:00 AM,1515,13,Newton,1324,2,946,OTHER MISCELLANEOUS CRIME,0344,0,M,H,103,ALLEY,NaN,NaN,IC,Invest Cont,946,NaN,NaN,NaN,1300 E 21ST ST,NaN,34.0224,-118.2524
3,090631215,2010 Jan 05 12:00:00 AM,2010 Jan 05 12:00:00 AM,0150,06,Hollywood,0646,2,900,VIOLATION OF COURT ORDER,1100 0400 1402,47,F,W,101,STREET,102,HAND GUN,IC,Invest Cont,900,998,NaN,NaN,CAHUENGA BL,HOLLYWOOD BL,34.1016,-118.3295
4,100100501,2010 Jan 03 12:00:00 AM,2010 Jan 02 12:00:00 AM,2100,01,Central,0176,1,122,"RAPE, ATTEMPTED",0400,47,F,H,103,ALLEY,400,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",IC,Invest Cont,122,NaN,NaN,NaN,8TH ST,SAN PEDRO ST,34.0387,-118.2488


In [4]:
# Key columns for the queries: inspect raw format + nulls
keys = ['DATE OCC', 'TIME OCC', 'Premis Desc', 'AREA NAME', 'LAT', 'LON']
for c in keys:
    print(f"\n=== {c} ===  nulls={crime[c].isna().sum()}")
    print(crime[c].dropna().head(4).tolist())


=== DATE OCC ===  nulls=0
['2010 Feb 20 12:00:00 AM', '2010 Sep 12 12:00:00 AM', '2010 Aug 09 12:00:00 AM', '2010 Jan 05 12:00:00 AM']

=== TIME OCC ===  nulls=0
['1350', '0045', '1515', '0150']

=== Premis Desc ===  nulls=11
['SINGLE FAMILY DWELLING', 'STREET', 'ALLEY', 'STREET']

=== AREA NAME ===  nulls=0
['Newton', 'Pacific', 'Newton', 'Hollywood']

=== LAT ===  nulls=0
['33.9825', '33.9599', '34.0224', '34.1016']

=== LON ===  nulls=0
['-118.2695', '-118.3962', '-118.2524', '-118.3295']


In [5]:
# DATE OCC format is e.g. '2010 Feb 20 12:00:00 AM'
# Spark to_date pattern 'yyyy MMM dd hh:mm:ss a'  ==  pandas '%Y %b %d %I:%M:%S %p'
parsed = pd.to_datetime(crime['DATE OCC'], format='%Y %b %d %I:%M:%S %p', errors='coerce')
print('unparsed fraction:', round(parsed.isna().mean(), 4))
print('date range:', parsed.min(), '->', parsed.max())
print('\nyear counts (sample):')
print(parsed.dt.year.value_counts().sort_index())

unparsed fraction: 0.0
date range: 2010-01-01 00:00:00 -> 2011-12-01 00:00:00

year counts (sample):
DATE OCC
2010    199981
2011        19
Name: count, dtype: int64


In [6]:
# TIME OCC is HHMM (military). Q1 buckets it into Morning/Afternoon/Evening/Night.
t = pd.to_numeric(crime['TIME OCC'], errors='coerce')
print('TIME OCC range:', t.min(), '->', t.max(), '| non-numeric:', t.isna().sum())
print('\nlen distribution (chars):')
print(crime['TIME OCC'].str.len().value_counts())
# Premis Desc == 'STREET' is the Q1 filter target
print("\nPremis Desc top values:")
print(crime['Premis Desc'].value_counts().head(10))
print("\n'STREET' present:", (crime['Premis Desc'] == 'STREET').sum())

TIME OCC range: 1 -> 2359 | non-numeric: 0

len distribution (chars):
TIME OCC
4    200000
Name: count, dtype: int64

Premis Desc top values:
Premis Desc
STREET                                          46478
SINGLE FAMILY DWELLING                          45203
MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)    26219
PARKING LOT                                     13349
OTHER BUSINESS                                   9664
SIDEWALK                                         9378
VEHICLE, PASSENGER/TRUCK                         7781
DRIVEWAY                                         3877
GARAGE/CARPORT                                   3228
DEPARTMENT STORE                                 2679
Name: count, dtype: int64

'STREET' present: 46478


In [7]:
# LAT/LON: floats, but LA crime data has (0,0) null-island sentinels -> filter in Q4
lat = pd.to_numeric(crime['LAT'], errors='coerce')
lon = pd.to_numeric(crime['LON'], errors='coerce')
print('LAT range:', lat.min(), lon.max())
print('LON range:', lon.min(), lon.max())
print('zero-island rows (LAT==0):', (lat == 0).sum(), '/', len(crime))

LAT range: 0.0 0.0
LON range: -118.7668 0.0
zero-island rows (LAT==0): 11 / 200000


In [8]:
# Confirm 2nd file has identical schema before unioning in Spark
crime_b = pd.read_csv(CRIME_B, nrows=5, dtype=str)
print('same columns:', list(crime.columns) == list(crime_b.columns))
if list(crime.columns) != list(crime_b.columns):
    print('A only:', set(crime.columns) - set(crime_b.columns))
    print('B only:', set(crime_b.columns) - set(crime.columns))

same columns: True


## 2. Police stations (Q4)

Tiny (21 rows). **BOM on the header** - first column reads as `﻿X` unless handled. In Spark we rename by position. `X`=lon, `Y`=lat.

In [9]:
# encoding='utf-8-sig' strips the BOM
stations = pd.read_csv(STATIONS, encoding='utf-8-sig')
print('shape:', stations.shape)
print('columns:', list(stations.columns))
print('dtypes:\n', stations.dtypes)
stations.head()

shape: (21, 6)
columns: ['X', 'Y', 'FID', 'DIVISION', 'LOCATION', 'PREC']
dtypes:
 X           float64
Y           float64
FID           int64
DIVISION        str
LOCATION        str
PREC          int64
dtype: object


,X,Y,FID,DIVISION,LOCATION,PREC
0,-118.289242,33.757661,1,HARBOR,2175 JOHN S. GIBSON BLVD.,5
1,-118.275394,33.938627,2,SOUTHEAST,145 W. 108TH ST.,18
2,-118.277670,33.970307,3,77TH STREET,7600 S. BROADWAY,12
3,-118.419842,33.991655,4,PACIFIC,12312 CULVER BLVD.,14
4,-118.305142,34.010575,5,SOUTHWEST,1546 MARTIN LUTHER KING JR. BLVD.,3


In [10]:
# Show what the BOM looks like if NOT stripped (why Spark renames by position)
raw = pd.read_csv(STATIONS)
print('first col name without utf-8-sig:', repr(raw.columns[0]))

first col name without utf-8-sig: 'X'


## 3. Income (Q3)

**Semicolon-delimited.** `Estimated Median Income` is a `$`/`,`-formatted string - must strip and cast to int. Some rows have null/empty income.

In [11]:
income = pd.read_csv(INCOME, sep=';', dtype=str)
print('shape:', income.shape)
print('columns:', list(income.columns))
income.head()

shape: (282, 3)
columns: ['Zip Code', 'Community', 'Estimated Median Income']


,Zip Code,Community,Estimated Median Income
0,90001,"Los Angeles (South Los Angeles), Florence-Graham","$52,806"
1,90002,"Los Angeles (Southeast Los Angeles, Watts)","$46,159"
2,90003,"Los Angeles (South Los Angeles, Southeast Los ...","$47,733"
3,90004,"Los Angeles (Hancock Park, Rampart Village, Vi...","$54,947"
4,90005,"Los Angeles (Hancock Park, Koreatown, Wilshire...","$44,913"


In [12]:
# Parse income: strip $ and , -> int  (mirrors regexp_replace in Datasets.income)
inc = income['Estimated Median Income'].str.replace(r'[$,]', '', regex=True)
inc_num = pd.to_numeric(inc, errors='coerce')
print('null/unparseable incomes:', inc_num.isna().sum(), '/', len(income))
print('income range:', inc_num.min(), '->', inc_num.max())
print('Zip Code sample:', income['Zip Code'].head(3).tolist())
print('zip dtype note: keep as string (leading zeros possible), 5-char')

null/unparseable incomes: 2 / 282
income range: 22291.0 -> 212115.0
Zip Code sample: ['90001', '90002', '90003']
zip dtype note: keep as string (leading zeros possible), 5-char


## 4. Census blocks (Q3)

GeoJSON, 171MB. **Physically one `Feature` per line** - so we read line-by-line in Spark (textFile -> filter -> json) to avoid OOM. We only need `properties.ZCTA20` (zip) and `properties.POP20` (population). Polygon geometry is ignored. First, the data dictionary:

In [13]:
fields = pd.read_csv(FIELDS)
print('dictionary rows:', len(fields))
fields

dictionary rows: 30


,Attribute,Data Type,Meaning
0,OBJECTID,OID,Object identifier (unique record ID)
1,State,String (2),"State FIPS code (""06"" for California)"
2,COUNTY,String (3),"County FIPS code (""037"" for Los Angeles County)"
3,CT20,String (6),Census tract number (6 digits)
4,BG20,String (7),Block group number (7 digits)
5,CB20,String (4),Census block number (4 digits)
6,CTCB20,String (10),Combined census tract and block identifier
7,FEAT_TYPE,String (20),"Feature type: land use classification (water, ..."
8,FIP20,String (5),Los Angeles County FIPS code
9,BGFIP20,String (12),Combined block group and FIPS code


In [14]:
# Confirm the 1-Feature-per-line layout + inspect one feature's properties
feat = None
with open(GEOJSON, 'r') as f:
    for i, line in enumerate(f):
        s = line.strip()
        if s.startswith('{ "type": "Feature"') or s.startswith('{"type":"Feature"'):
            feat = json.loads(s.rstrip(','))
            break
        if i > 5:
            print('first lines (header):')
            break
print('keys:', list(feat.keys()))
print('property keys:', list(feat['properties'].keys()))
print('\nZCTA20 =', feat['properties'].get('ZCTA20'), '| POP20 =', feat['properties'].get('POP20'))
print('geometry type:', feat['geometry']['type'])

keys: ['type', 'properties', 'geometry']
property keys: ['OBJECTID', 'State', 'COUNTY', 'CT20', 'BG20', 'CB20', 'CTCB20', 'FEAT_TYPE', 'FIP20', 'BGFIP20', 'CITY', 'COMM', 'CITYCOMM', 'ZCTA20', 'HD22', 'HD_NAME', 'SPA22', 'SPA_NAME', 'SUP21', 'SUP_LABEL', 'POP20', 'HOUSING20', 'FIP_CURRENT', 'BG20FIP_CURRENT', 'CITY_CURRENT', 'COMM_CURRENT', 'CITYCOMM_CURRENT', 'ShapeSTArea', 'ShapeSTLength']

ZCTA20 = 91344 | POP20 = 80
geometry type: Polygon


In [15]:
# Aggregate population by zip (full pass, but only parse properties -> light).
# Mirrors Datasets.censusPopulation: group POP20 by ZCTA20.
from collections import defaultdict
pop_by_zip = defaultdict(int)
blocks = 0
with open(GEOJSON, 'r') as f:
    for line in f:
        s = line.strip()
        if not (s.startswith('{ "type": "Feature"') or s.startswith('{"type":"Feature"')):
            continue
        props = json.loads(s.rstrip(','))['properties']
        z, p = props.get('ZCTA20'), props.get('POP20')
        if z is not None and p is not None:
            pop_by_zip[str(z)] += int(p)
            blocks += 1
print('feature blocks:', blocks)
print('distinct zips:', len(pop_by_zip))
print('total population:', sum(pop_by_zip.values()))
print('\ntop 5 zips by pop:')
for z, p in sorted(pop_by_zip.items(), key=lambda kv: -kv[1])[:5]:
    print(f'  {z}: {p:,}')

feature blocks: 90970
distinct zips: 304
total population: 10013638

top 5 zips by pop:
  90650: 102,891
  90011: 102,308
  91331: 100,720
  90250: 97,653
  91342: 96,429


## 5. Spark read summary

Confirmed read recipes (already in `gr.ntua.dsml.common.Datasets`):

- **Crime**: `spark.read.option("header",true).csv(both files)` then `union`. Cast `TIME OCC`→int, `LAT`/`LON`→double, `to_date("DATE OCC", "yyyy MMM dd hh:mm:ss a")`. Q1 filter `Premis Desc = 'STREET'`. Q4 drop `LAT==0` sentinels.
- **Police stations**: `header=true`, rename by position (BOM on `X`). `X`=lon, `Y`=lat.
- **Income**: `option("sep", ";")`, `regexp_replace(income, "[$,]", "").cast(int)`, keep `Zip Code` as string.
- **Census**: `textFile` → filter `Feature` lines → strip trailing comma → `spark.read.json(Dataset<String>)` → `properties.ZCTA20`,`properties.POP20`. (multiLine=true OOMs at 2GB.)
- **Join keys**: census `ZCTA20` ↔ income `Zip Code` (both 5-char string zips).